# BCARS VST denoising — all paper metrics

Recomputes/reports every number quoted in the paper from the Zenodo deposit.

| Section | Paper item |
|---|---|
| 1 | Table 1 — simulated benchmark, 7 methods × (whole / foreground / background) × (PSNR, SSIM) |
| 2 | Glycerol vs. measurement-based pseudo-ground-truth |
| 3 | Bead FWHM (Fig. 4b) — refitted here from the deposited Raman crops |
| 4 | Pipeline check — runs the released VST→CCV→phase-retrieval code on the deposited glycerol cube |

**Data.** Download the Zenodo deposit and either unzip it next to this notebook as `data/`,
or set `BCARS_DATA_ROOT` to the folder containing `raman/`, `metrics/` and `inputs/`.

The deposit contains the retrieved **Raman** cubes per method and the metric tables.
Baseline denoiser code (DDS2M, DIP, N2N, BM4D) is not part of this repository — see
the README for where it lives and the settings used.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "bcars_processing").is_dir())
DATA = Path(os.environ.get("BCARS_DATA_ROOT", REPO / "data"))

RAMAN   = DATA / "raman"
METRICS = DATA / "metrics"
INPUTS  = DATA / "inputs"
print("code:", REPO, "\ndata:", DATA)
for p in (RAMAN, METRICS, INPUTS):
    print(("  ok      " if p.exists() else "  MISSING ") + p.name)

code: /mnt/d/GaTech Dropbox/Haoyu Xu/workspace/crikit3_denoise/bcars-vst-denoise 
data: /tmp/claude-1000/-mnt-d-GaTech-Dropbox-Haoyu-Xu-workspace-crikit3-denoise/4d281a12-e186-4d8a-a816-79644f2ef4ac/scratchpad/zenodo_test
  ok      raman
  ok      metrics
  ok      inputs


## 1. Table 1 — simulated benchmark

Five held-out simulated FOVs (AD_03, AD_05, l4_04, l4_05, l4_06) with exact peak-fit ground
truth. Each denoised cube is affine-aligned to the ground truth, then scored over the whole
image and over foreground/background masks (threshold → largest connected component → convex
hull of the ground-truth spectral sum). Mean ± std across the five FOVs.

In [2]:
ORDER = ["DDS2M", "DIP", "N2N", "BM4D", "CCV", "SVD", "Noisy"]
canon = lambda s: str(s).split("(")[0].strip()      # 'SVD(r=7)' -> 'SVD'

def table1(csv_path):
    df = pd.read_csv(csv_path)
    df["m"] = df["method"].map(canon)
    df = df.set_index("m")
    rows = [{
        "Method": m,
        "PSNR whole": "%.2f ± %.2f" % (df.loc[m].psnr_full_mean, df.loc[m].psnr_full_std),
        "SSIM whole": "%.3f ± %.3f" % (df.loc[m].ssim_full_mean, df.loc[m].ssim_full_std),
        "PSNR fg":    "%.2f ± %.2f" % (df.loc[m].psnr_fg_mean,   df.loc[m].psnr_fg_std),
        "SSIM fg":    "%.3f ± %.3f" % (df.loc[m].ssim_fg_mean,   df.loc[m].ssim_fg_std),
        "PSNR bg":    "%.2f ± %.2f" % (df.loc[m].psnr_bg_mean,   df.loc[m].psnr_bg_std),
        "SSIM bg":    "%.3f ± %.3f" % (df.loc[m].ssim_bg_mean,   df.loc[m].ssim_bg_std),
    } for m in ORDER if m in df.index]
    return pd.DataFrame(rows).set_index("Method")

t_raman = table1(METRICS / "sim_table1/overall_metrics_abs_meanstd.csv")   # Im chi_R = paper Table 1
print("Raman (Im chi_R) lineshapes — paper Table 1")
display(t_raman)

print("\nDispersive (Re chi) lineshapes — same cubes, SI")
display(table1(METRICS / "sim_table1/overall_metrics_disp_meanstd.csv"))

Raman (Im chi_R) lineshapes — paper Table 1


,PSNR whole,SSIM whole,PSNR fg,SSIM fg,PSNR bg,SSIM bg
Method,,,,,,
DDS2M,41.43 ± 0.69,0.475 ± 0.041,39.90 ± 0.85,0.523 ± 0.060,42.40 ± 1.15,0.434 ± 0.034
DIP,39.96 ± 0.83,0.488 ± 0.056,40.14 ± 0.76,0.556 ± 0.064,39.67 ± 1.16,0.440 ± 0.038
N2N,39.62 ± 0.93,0.382 ± 0.034,37.69 ± 0.40,0.393 ± 0.045,41.32 ± 1.75,0.373 ± 0.024
BM4D,39.46 ± 1.09,0.345 ± 0.035,37.78 ± 0.70,0.353 ± 0.049,40.81 ± 1.71,0.337 ± 0.022
CCV,39.20 ± 1.00,0.455 ± 0.056,38.78 ± 1.44,0.556 ± 0.053,39.22 ± 1.10,0.382 ± 0.046
SVD,37.86 ± 1.40,0.396 ± 0.078,37.65 ± 1.53,0.483 ± 0.079,37.86 ± 1.45,0.339 ± 0.047
Noisy,29.30 ± 1.21,0.127 ± 0.024,26.61 ± 0.58,0.110 ± 0.010,32.24 ± 3.16,0.133 ± 0.029



Dispersive (Re chi) lineshapes — same cubes, SI


,PSNR whole,SSIM whole,PSNR fg,SSIM fg,PSNR bg,SSIM bg
Method,,,,,,
DDS2M,43.27 ± 0.64,0.561 ± 0.039,41.96 ± 0.87,0.612 ± 0.054,44.05 ± 1.01,0.520 ± 0.034
DIP,42.94 ± 0.94,0.593 ± 0.054,42.75 ± 1.00,0.670 ± 0.065,42.89 ± 1.02,0.540 ± 0.034
N2N,41.82 ± 0.91,0.494 ± 0.028,40.03 ± 0.44,0.496 ± 0.043,43.31 ± 1.62,0.488 ± 0.021
BM4D,41.64 ± 1.08,0.466 ± 0.032,40.08 ± 0.75,0.465 ± 0.053,42.83 ± 1.61,0.461 ± 0.020
CCV,41.54 ± 1.33,0.573 ± 0.060,40.80 ± 1.60,0.660 ± 0.060,41.81 ± 1.50,0.510 ± 0.046
SVD,40.60 ± 1.48,0.518 ± 0.080,40.11 ± 1.53,0.604 ± 0.081,40.82 ± 1.57,0.465 ± 0.049
Noisy,30.43 ± 1.53,0.201 ± 0.041,27.58 ± 0.86,0.192 ± 0.026,33.81 ± 3.79,0.201 ± 0.051


In [3]:
# LaTeX body for Table 1
for m, r in t_raman.iterrows():
    cells = [r[c].replace(" ± ", " \\pm ") for c in
             ["PSNR whole", "SSIM whole", "PSNR fg", "SSIM fg", "PSNR bg", "SSIM bg"]]
    print("%-6s & " % m + " & ".join("$%s$" % c for c in cells) + " \\\\")

# per-FOV values behind the means
per = sorted((METRICS / "sim_table1/per_fov").glob("*_abs.csv"))
print("\nper-FOV files:", ", ".join(p.stem for p in per))

DDS2M  & $41.43 \pm 0.69$ & $0.475 \pm 0.041$ & $39.90 \pm 0.85$ & $0.523 \pm 0.060$ & $42.40 \pm 1.15$ & $0.434 \pm 0.034$ \\
DIP    & $39.96 \pm 0.83$ & $0.488 \pm 0.056$ & $40.14 \pm 0.76$ & $0.556 \pm 0.064$ & $39.67 \pm 1.16$ & $0.440 \pm 0.038$ \\
N2N    & $39.62 \pm 0.93$ & $0.382 \pm 0.034$ & $37.69 \pm 0.40$ & $0.393 \pm 0.045$ & $41.32 \pm 1.75$ & $0.373 \pm 0.024$ \\
BM4D   & $39.46 \pm 1.09$ & $0.345 \pm 0.035$ & $37.78 \pm 0.70$ & $0.353 \pm 0.049$ & $40.81 \pm 1.71$ & $0.337 \pm 0.022$ \\
CCV    & $39.20 \pm 1.00$ & $0.455 \pm 0.056$ & $38.78 \pm 1.44$ & $0.556 \pm 0.053$ & $39.22 \pm 1.10$ & $0.382 \pm 0.046$ \\
SVD    & $37.86 \pm 1.40$ & $0.396 \pm 0.078$ & $37.65 \pm 1.53$ & $0.483 \pm 0.079$ & $37.86 \pm 1.45$ & $0.339 \pm 0.047$ \\
Noisy  & $29.30 \pm 1.21$ & $0.127 \pm 0.024$ & $26.61 \pm 0.58$ & $0.110 \pm 0.010$ & $32.24 \pm 3.16$ & $0.133 \pm 0.029$ \\

per-FOV files: AD_03_abs, AD_05_abs, l4_04_abs, l4_05_abs, l4_06_abs


## 2. Glycerol vs. pseudo-ground-truth

Glycerol is spatially homogeneous, so averaging the noisy spectra along the slow scan axis
gives a measurement-based reference (`galvo`). `fovmean` and `galvo_ho` (held-out strip) are
sensitivity checks on that choice.

In [4]:
gly = pd.read_csv(METRICS / "glycerol/metrics_raman_galvo.csv").set_index("method")
print("Glycerol, Raman domain, vs galvo pseudo-GT")
display(gly[["psnr_overall", "ssim_overall", "sam_mean_rad", "pearson_mean"]].round(4))

print("\nPSNR under each choice of reference")
display(pd.read_csv(METRICS / "glycerol/psnr_by_reference_raman.csv", index_col=0).round(2))

Glycerol, Raman domain, vs galvo pseudo-GT


,psnr_overall,ssim_overall,sam_mean_rad,pearson_mean
method,,,,
BM4D,32.9695,0.1062,0.0838,0.9921
DDS2M,31.5489,0.0315,0.1044,0.9865
DIP,30.7583,0.0385,0.1149,0.9846
SVD(r=7),30.1626,0.0313,0.1173,0.9836
CCV(tau0.5),29.9743,0.0370,0.1199,0.9831
N2N,28.2120,0.0211,0.1652,0.9698
Noisy,20.4640,0.0240,0.4464,0.7737



PSNR under each choice of reference


,fovmean,galvo,galvo_ho,d_galvo_minus_fovmean,d_holdout_minus_galvo
BM4D,30.40,32.97,31.23,2.57,-1.74
DDS2M,50.12,31.55,29.92,-18.57,-1.63
DIP,26.95,30.76,30.99,3.81,0.23
SVD(r=7),26.88,30.16,30.34,3.29,0.17
CCV(tau0.5),26.73,29.97,30.12,3.24,0.14
N2N,25.49,28.21,28.67,2.72,0.46
Noisy,18.72,20.46,21.03,1.75,0.56


## 3. Bead FWHM (Fig. 4b)

Single-band Raman images of a 1 µm polystyrene bead: 17×17 crop, 260 nm pixel pitch. A 1-D
Gaussian is fitted to the horizontal and vertical profiles through the bead centre; the centre
is taken from the BM4D peak and shared by all methods. FWHM = 2√(2 ln2)·σ, averaged over the
two directions. Refitted here from the deposited crops and cross-checked against the table.

In [5]:
import tifffile as tiff
from scipy.optimize import curve_fit

PIXEL_NM, BAND_IDX, FWHM_K = 260.0, 318, 2.3548200450309493

cubes, wn = {}, None
for name in ["Noisy", "SVD", "CCV", "BM4D", "N2N", "DDS2M", "DIP", "N2N_old"]:
    fp = RAMAN / "bead_crop" / ("raman_bead_%s.tif" % name)
    if not fp.exists():
        continue
    with tiff.TiffFile(str(fp)) as tf:
        cubes[name] = tf.asarray().astype(np.float32)
        if wn is None:
            wn = np.array([float(l.split()[0]) for l in tf.imagej_metadata["Labels"]])

band = {k: c[BAND_IDX] for k, c in cubes.items()}
cy, cx = np.unravel_index(int(np.argmax(band["BM4D"])), band["BM4D"].shape)
print("band %d = %.1f cm^-1 | bead centre (row, col) = (%d, %d)" % (BAND_IDX, wn[BAND_IDX], cy, cx))

def gauss1d(x, amp, mu, sigma, off):
    return off + amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def fwhm_px(profile, centre):
    x = np.arange(profile.size, dtype=float)
    p0 = [max(profile.max() - profile.min(), 1e-6), float(centre), 1.5, profile.min()]
    bounds = ([0.0, centre - 2.5, 0.3, -np.inf], [np.inf, centre + 2.5, 8.0, np.inf])
    try:
        popt, _ = curve_fit(gauss1d, x, profile, p0=p0, bounds=bounds, maxfev=5000)
        return abs(popt[2]) * FWHM_K
    except RuntimeError:
        return np.nan

rows = []
for name, m in band.items():
    fh = fwhm_px(m[cy, :].astype(float), cx)
    fv = fwhm_px(m[:, cx].astype(float), cy)
    rows.append({"Method": name, "FWHM_h (nm)": fh * PIXEL_NM, "FWHM_v (nm)": fv * PIXEL_NM,
                 "FWHM mean (nm)": np.nanmean([fh, fv]) * PIXEL_NM})
fwhm = pd.DataFrame(rows).set_index("Method").round(0)

ref = pd.read_csv(METRICS / "bead/bead_fwhm_summary.csv").set_index("method")["fwhm_avg_nm"]
fwhm["deposited (nm)"] = ref.round(0)
display(fwhm)

band 318 = 1016.2 cm^-1 | bead centre (row, col) = (8, 8)


,FWHM_h (nm),FWHM_v (nm),FWHM mean (nm),deposited (nm)
Method,,,,
Noisy,731.0,584.0,658.0,658.0
SVD,731.0,574.0,652.0,652.0
CCV,749.0,568.0,658.0,658.0
BM4D,747.0,571.0,659.0,659.0
N2N,670.0,572.0,621.0,621.0
DDS2M,1823.0,1328.0,1575.0,1575.0
DIP,796.0,599.0,698.0,698.0
N2N_old,1467.0,683.0,1075.0,1075.0


## 4. Pipeline check — released code on the deposited glycerol cube

Runs the code in `bcars_processing/` (CCV rank selection → symmetric reconstruction → Hilbert
phase retrieval → phase-error correction) on the deposited noisy glycerol cube, and compares
the result with the deposited `raman_gly_CCV.tif`. This verifies the released pipeline
reproduces the CCV column of the figures.

In [6]:
import yaml, scipy.io as sio
sys.path.insert(0, str(REPO / "bcars_processing/ccvsvd"))
sys.path.insert(0, str(REPO / "bcars_processing/kk_pec"))
from SVD_utils import compute_crosscov_svd
from criteria import select_by_corr, RECON
from postprocess import PostprocessConfig, retrieve_phase
from crikit.cri.error_correction import PhaseErrCorrectALS

prm = yaml.safe_load(open(REPO / "bcars_processing/params.yaml"))
mat = sio.loadmat(INPUTS / "preprocessed_nofilter_glycerol_4Xdiluted_20251111.mat")
cube_n = np.asarray(mat["y_0_real"], dtype=np.float64)
nmin, nmax = float(np.ravel(mat["norm_min"])[0]), float(np.ravel(mat["norm_max"])[0])
cube = cube_n * (nmax - nmin) + nmin                      # back to physical (dispersive VST)

cc = compute_crosscov_svd(cube)
keep, r = select_by_corr(cc, tau=prm["ccvsvd"]["tau"])
den = RECON[prm["ccvsvd"]["recon"]](cc, cube, keep, smooth=prm["ccvsvd"]["smooth"])
print("CCV kept %d / %d components (tau=%.2f)" % (keep.size, r.size, prm["ccvsvd"]["tau"]))

pp = prm["postprocess"]
cfg = PostprocessConfig(phase_method=pp["phase_method"],
                        hilbert_pad_factor=pp["hilbert"]["pad_factor"],
                        hilbert_n_edge=pp["hilbert"]["n_edge"],
                        phase_err_correct=pp["pec"]["enabled"],
                        pec_kwargs={k: pp["pec"][k] for k in
                                    ("smoothness_param", "asym_param", "redux", "order",
                                     "max_iter", "min_diff", "verbose")})
spec = retrieve_phase(den, cfg)                            # same chain as step2_process.py
if cfg.phase_err_correct:
    spec = PhaseErrCorrectALS(**cfg.pec_kwargs).calculate(spec)
raman_here = np.imag(spec)

dep = tiff.imread(str(RAMAN / "glycerol/raman_gly_CCV.tif")).astype(np.float32)  # (wn, y, x)
dep = np.transpose(dep, (1, 2, 0))
n = min(raman_here.shape[-1], dep.shape[-1])
a, b = raman_here[..., :n].ravel(), dep[..., :n].ravel()
print("shapes: recomputed %s vs deposited %s" % (raman_here.shape, dep.shape))
print("Pearson r vs deposited CCV cube: %.4f" % np.corrcoef(a, b)[0, 1])
print("(~0.99: the figure pipeline ran CCV in the normalized domain, this runs it in the\n physical domain; phase-error correction is nonlinear, so the two differ slightly.)")

CCV kept 19 / 342 components (tau=0.50)
shapes: recomputed (64, 64, 685) vs deposited (64, 64, 685)
Pearson r vs deposited CCV cube: 0.9900
(~0.99: the figure pipeline ran CCV in the normalized domain, this runs it in the
 physical domain; phase-error correction is nonlinear, so the two differ slightly.)


### 4b. Same check on a simulated FOV

The simulated cubes were phase-retrieved with Kramers-Kronig (settings in each FOV's
`meta.json`), not the Hilbert default of `params.yaml`, so the config is set explicitly here.
On simulated data the deposited CCV cube is reproduced exactly.


In [7]:
fov = "l4_05"   # smallest FOV (192x192); any of the five works
m = sio.loadmat(INPUTS / ("simulated/%s.mat" % fov), variable_names=["y_0_real"])
Z = np.asarray(m["y_0_real"], dtype=np.float64)     # noisy VST cube, normalized [0,1]

cc = compute_crosscov_svd(Z)
keep, r = select_by_corr(cc, tau=0.5)
den = RECON["symmetric"](cc, Z, keep, smooth=0.0)
print("CCV kept %d / %d components" % (keep.size, r.size))

cfg_sim = PostprocessConfig(
    phase_method="kk", cars_amp_offset=1.0, nrb_amp_offset=1.0, conjugate=False,
    phase_offset=0.0, norm_to_nrb=True, pad_factor=1, n_edge=30, bad_value=1e-8,
    phase_err_correct=True,
    pec_kwargs=dict(smoothness_param=1, asym_param=1e-5, redux=5, order=2,
                    max_iter=100, min_diff=1e-6, verbose=False))
sp = retrieve_phase(den, cfg_sim)
sp = PhaseErrCorrectALS(**cfg_sim.pec_kwargs).calculate(sp)

dep = np.transpose(tiff.imread(str(RAMAN / ("sim_%s/raman_CCV.tif" % fov))).astype(np.float32), (1, 2, 0))
print("Pearson r vs deposited CCV cube: %.5f" % np.corrcoef(np.imag(sp).ravel(), dep.ravel())[0, 1])

CCV kept 2 / 347 components
Pearson r vs deposited CCV cube: 1.00000


## Where each number appears

- **Section 1** → Table 1 (`tab:tsim`) and the PSNR/SSIM gains quoted in Results.
- **Section 2** → the glycerol PSNR quoted in Results and the SI glycerol table.
- **Section 3** → Fig. 4(b) and the FWHM values in Results.
- **Section 4** → the CCV column of Figs. 3–5; also a smoke test of the released pipeline.

Denoiser settings for every method are in the README; the deposit ships their outputs, not
their code.